In [10]:
import os
import sys
import glob
import re
import datetime
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import holidays

plt.rcParams['font.sans-serif'] = ['DejaVu Sans', 'Arial', 'Calibri', 'Segoe UI', 'sans-serif']
plt.rcParams['axes.unicode_minus'] = False


# ============================================================
# 1. СПИСОК ПРАЗДНИКОВ РОССИИ
# ============================================================
def get_russia_holidays(years=None):
    """Возвращает dict {date: name} праздников России."""
    if years is None:
        years = range(2023, 2027)

    ru_holidays = {}

    # Федеральные праздники (фиксированные даты)
    fixed_holidays = {
        (1, 1):  "Новый год",
        (1, 2):  "Новогодние каникулы",
        (1, 3):  "Новогодние каникулы",
        (1, 4):  "Новогодние каникулы",
        (1, 5):  "Новогодние каникулы",
        (1, 6):  "Новогодние каникулы",
        (1, 7):  "Рождество Христово",
        (1, 8):  "Новогодние каникулы",
        (2, 23): "День защитника Отечества",
        (3, 8):  "Международный женский день",
        (5, 1):  "Праздник Весны и Труда",
        (5, 9):  "День Победы",
        (6, 12): "День России",
        (11, 4): "День народного единства",
    }

    for year in years:
        for (m, d), name in fixed_holidays.items():
            try:
                dt = datetime.date(year, m, d)
                ru_holidays[dt] = name
            except ValueError:
                pass

        # Переносы выходных
        for (m, d), name in fixed_holidays.items():
            try:
                dt = datetime.date(year, m, d)
                if dt.weekday() == 5:  # суббота
                    next_monday = dt + datetime.timedelta(days=2)
                    if next_monday not in ru_holidays:
                        ru_holidays[next_monday] = f"{name} (перенос)"
                elif dt.weekday() == 6:  # воскресенье
                    next_monday = dt + datetime.timedelta(days=1)
                    if next_monday not in ru_holidays:
                        ru_holidays[next_monday] = f"{name} (перенос)"
            except ValueError:
                pass

    # Используем библиотеку holidays для точных данных
    try:
        ru_official = holidays.RU(years=years, subdiv='SAR')
        for dt, name in ru_official.items():
            if isinstance(dt, datetime.date):
                ru_holidays[dt] = name
            else:
                ru_holidays[datetime.date.fromisoformat(str(dt))] = name
    except Exception:
        try:
            ru_official = holidays.Russia(years=years)
            for dt, name in ru_official.items():
                if isinstance(dt, datetime.date):
                    ru_holidays[dt] = name
                else:
                    ru_holidays[datetime.date.fromisoformat(str(dt))] = name
        except Exception:
            pass

    return ru_holidays


# ============================================================
# 2. ПОТОКОВЫЙ ПАРСЕР EXCEL
# ============================================================
def parse_traffic_excel(filepath):
    """
    Потоковый парсер Excel-файла с данными трафика.
    Распознаёт два участка дороги по заголовкам.
    Игнорирует строки "Итого", "Среднее", "%".
    """
    wb = openpyxl.load_workbook(filepath, read_only=True, data_only=True)
    ws = wb.active

    records = []
    current_section = None
    section_count = 0

    # Regex для распознавания заголовков участков
    section_regex = re.compile(
        r'(?:Р-22\s*"Каспий"[^к]*км\s*\d+\+\d+|км\s*\d+\+\d+\s*а/д\s*Р-22)',
        re.IGNORECASE
    )

    # Regex для дат в формате "01.09.2023 00:59:59"
    date_regex = re.compile(r'^\s*\d{1,2}\.\d{1,2}\.\d{4}\s+\d{1,2}:\d{2}(:\d{2})?\s*$')

    # Ключевые слова для строк-итогов (которые нужно пропускать)
    skip_keywords = {'итого', 'среднее', '%', 'процент'}

    row_num = 0
    for row in ws.iter_rows(values_only=True):
        row_num += 1
        if not row or all(cell is None for cell in row):
            continue

        first_val = row[0]
        if first_val is None:
            continue

        first_str = str(first_val).strip().lower()

        # Пропускаем строки "Итого", "Среднее", "%"
        if any(keyword in first_str for keyword in skip_keywords):
            continue

        # Проверяем, является ли это заголовком участка
        is_section_header = False
        for cell in row[:8]:
            if cell is not None and isinstance(cell, str):
                cell_s = cell.strip()
                if section_regex.search(cell_s):
                    current_section = cell_s
                    section_count += 1
                    is_section_header = True
                    break
                elif ('р-22' in cell_s.lower() or 'касмпий' in cell_s.lower()) and len(cell_s) > 20:
                    current_section = cell_s
                    section_count += 1
                    is_section_header = True
                    break

        if is_section_header:
            continue

        # Проверяем, является ли первая ячейка датой
        dt_obj = None
        if isinstance(first_val, (datetime.datetime, datetime.date)):
            dt_obj = pd.to_datetime(first_val)
        elif date_regex.match(str(first_val).strip()):
            try:
                dt_obj = pd.to_datetime(first_val, dayfirst=True)
            except Exception:
                dt_obj = None

        if dt_obj is not None and pd.notnull(dt_obj) and current_section is not None:
            row_vals = list(row[1:29])
            if len(row_vals) < 28:
                row_vals.extend([0] * (28 - len(row_vals)))

            records.append([current_section, dt_obj] + row_vals[:28])

    wb.close()

    col_names = [
        'road_section', 'timestamp',
        'vol_total', 'vol_direct', 'vol_reverse',
        'cars_total', 'cars_direct', 'cars_reverse',
        'small_trucks_total', 'small_trucks_direct', 'small_trucks_reverse',
        'med_trucks_total', 'med_trucks_direct', 'med_trucks_reverse',
        'large_trucks_total', 'large_trucks_direct', 'large_trucks_reverse',
        'road_trains_total', 'road_trains_direct', 'road_trains_reverse',
        'buses_total', 'buses_direct', 'buses_reverse',
        'motos_total', 'motos_direct', 'motos_reverse',
        'speed_direct', 'speed_reverse',
        'load_direct', 'load_reverse'
    ]

    if not records:
        return pd.DataFrame(columns=col_names)

    df = pd.DataFrame(records, columns=col_names)

    numeric_cols = col_names[2:]
    for col in numeric_cols:
        if df[col].dtype == object:
            df[col] = (df[col].astype(str)
                       .str.replace('\xa0', '', regex=False)
                       .str.replace(' ', '', regex=False)
                       .str.replace(',', '.', regex=False))
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

    start_date = pd.Timestamp('2023-09-01')
    end_date = pd.Timestamp('2026-09-01')
    df = df[(df['timestamp'] >= start_date) & (df['timestamp'] <= end_date)]

    print(f"   Загружено {len(df)} записей для {section_count} участков")
    return df


# ============================================================
# 3. ДЕТЕКТОР АНОМАЛИЙ (СУТОЧНЫЕ)
# ============================================================
def detect_daily_anomalies(df, holidays_dict):
    """Поиск аномалий в суточных данных."""
    if df.empty:
        return pd.DataFrame()

    df = df.copy()
    df['date'] = df['timestamp'].dt.date
    df['hour'] = df['timestamp'].dt.hour
    df['is_weekend'] = df['timestamp'].dt.weekday.isin([5, 6]).astype(int)
    df['month_key'] = df['timestamp'].dt.to_period('M').astype(str)
    df['holiday_name'] = df['date'].map(holidays_dict)

    anomalies_list = []

    for section, sec_df in df.groupby('road_section'):
        print(f"\n   Анализ участка: {section[:60]}...")

        for dt_date, day_df in sec_df.groupby('date'):
            day_df = day_df.sort_values('hour')
            hours_cnt = len(day_df)

            # 1. Проверка полноты суток
            if hours_cnt < 24:
                anomalies_list.append({
                    'Участок дороги': section,
                    'Дата': dt_date,
                    'Уровень': 'Критический',
                    'Детектор': 'Целостность данных',
                    'Тип аномалии': 'Неполные сутки',
                    'Параметр': 'Часов в сутках',
                    'Факт': hours_cnt,
                    'Норма': 24,
                    'Отклонение (%)': round((hours_cnt - 24) / 24 * 100, 1),
                    'Индекс (Score)': 3.0,
                    'Причина': _get_cause(dt_date, holidays_dict, day_df),
                    'Описание цифрами': f"В сутках сохранено лишь {hours_cnt} часов вместо 24"
                })

            # 2. Залипание датчика
            vols = day_df['vol_total'].values
            max_consec = 1
            cur_consec = 1
            flat_val = None
            for i in range(1, len(vols)):
                if abs(vols[i] - vols[i-1]) < 0.01 and vols[i] > 5:
                    cur_consec += 1
                    if cur_consec > max_consec:
                        max_consec = cur_consec
                        flat_val = vols[i]
                else:
                    cur_consec = 1

            if max_consec >= 4:
                anomalies_list.append({
                    'Участок дороги': section,
                    'Дата': dt_date,
                    'Уровень': 'Критический',
                    'Детектор': 'Датчик',
                    'Тип аномалии': 'Залипание датчика (Flatline)',
                    'Параметр': 'Интенсивность',
                    'Факт': int(flat_val),
                    'Норма': 'Вариативность',
                    'Отклонение (%)': 0.0,
                    'Индекс (Score)': 4.0,
                    'Причина': _get_cause(dt_date, holidays_dict, day_df),
                    'Описание цифрами': f"Значение не менялось {max_consec} часов подряд ({int(flat_val)} авт./ч)"
                })

            # 3. Дисбаланс направлений
            daytime_df = day_df[(day_df['hour'] >= 7) & (day_df['hour'] <= 21)]
            if len(daytime_df) > 0:
                daytime_tot = daytime_df['vol_total'].sum()
                if daytime_tot > 500:
                    dir_ratio = daytime_df['vol_direct'].sum() / daytime_tot
                    if dir_ratio > 0.80 or dir_ratio < 0.20:
                        anomalies_list.append({
                            'Участок дороги': section,
                            'Дата': dt_date,
                            'Уровень': 'Высокий',
                            'Детектор': 'Направление',
                            'Тип аномалии': 'Дисбаланс направлений',
                            'Параметр': 'Доля прямого напр.',
                            'Факт': round(dir_ratio * 100, 1),
                            'Норма': '40-60%',
                            'Отклонение (%)': round((dir_ratio - 0.5) * 100, 1),
                            'Индекс (Score)': 2.8,
                            'Причина': _get_cause(dt_date, holidays_dict, day_df),
                            'Описание цифрами': f"Доля прямого направления составила {dir_ratio*100:.1f}% от дневного объема"
                        })

            # 4. Экстремальный затор
            if len(daytime_df) > 0:
                min_spd = daytime_df['speed_direct'].min()
                max_load = daytime_df['load_direct'].max()
                if min_spd < 30 and max_load > 70:
                    jam_h = daytime_df.loc[daytime_df['speed_direct'] == min_spd, 'hour'].iloc[0]
                    anomalies_list.append({
                        'Участок дороги': section,
                        'Дата': dt_date,
                        'Уровень': 'Критический',
                        'Детектор': 'Скорость/Загрузка',
                        'Тип аномалии': 'Экстремальный затор',
                        'Параметр': 'Мин. скорость (Прямое)',
                        'Факт': round(min_spd, 1),
                        'Норма': '> 60 км/ч',
                        'Отклонение (%)': round((min_spd - 80) / 80 * 100, 1),
                        'Индекс (Score)': 4.5,
                        'Причина': _get_cause(dt_date, holidays_dict, day_df),
                        'Описание цифрами': f"В {int(jam_h)}:00 скорость упала до {min_spd} км/ч при загрузке {max_load}%"
                    })

            # 5. Аномальный спад/всплеск интенсивности
            m_key = day_df['month_key'].iloc[0]
            is_wknd = day_df['is_weekend'].iloc[0]
            ref_df = df[(df['road_section'] == section) &
                       (df['month_key'] == m_key) &
                       (df['is_weekend'] == is_wknd)]

            if len(ref_df) > 5:
                daily_vols_ref = ref_df.groupby('date')['vol_total'].sum()
                vol_med = daily_vols_ref.median()
                vol_mad = (daily_vols_ref - vol_med).abs().median()
                vol_mad = vol_mad if vol_mad > 0 else daily_vols_ref.std()
                vol_mad = vol_mad if (vol_mad > 0 and pd.notnull(vol_mad)) else 1.0

                day_vol = day_df['vol_total'].sum()
                z_vol = 0.6745 * (day_vol - vol_med) / vol_mad if vol_mad > 0 else 0
                delta_p = round((day_vol - vol_med) / vol_med * 100, 1) if vol_med > 0 else 0

                if (z_vol < -2.5 and delta_p <= -30.0) or delta_p <= -50.0:
                    anomalies_list.append({
                        'Участок дороги': section,
                        'Дата': dt_date,
                        'Уровень': 'Критический',
                        'Детектор': 'Амплитуда',
                        'Тип аномалии': 'Аномальный спад интенсивности',
                        'Параметр': 'Суточный объем',
                        'Факт': int(day_vol),
                        'Норма': int(vol_med),
                        'Отклонение (%)': delta_p,
                        'Индекс (Score)': round(abs(z_vol), 1),
                        'Причина': _get_cause(dt_date, holidays_dict, day_df),
                        'Описание цифрами': f"Суточный трафик {int(day_vol)} ниже нормы {int(vol_med)} на {abs(delta_p)}% (Z={z_vol:.1f})"
                    })
                elif (z_vol > 2.5 and delta_p >= 30.0) or delta_p >= 50.0:
                    anomalies_list.append({
                        'Участок дороги': section,
                        'Дата': dt_date,
                        'Уровень': 'Высокий',
                        'Детектор': 'Амплитуда',
                        'Тип аномалии': 'Аномальный всплеск интенсивности',
                        'Параметр': 'Суточный объем',
                        'Факт': int(day_vol),
                        'Норма': int(vol_med),
                        'Отклонение (%)': delta_p,
                        'Индекс (Score)': round(abs(z_vol), 1),
                        'Причина': _get_cause(dt_date, holidays_dict, day_df),
                        'Описание цифрами': f"Суточный трафик {int(day_vol)} выше нормы {int(vol_med)} на +{delta_p}% (Z={z_vol:.1f})"
                    })

            # 6. Искажение суточного профиля
            if hours_cnt == 24:
                norm_hourly_vol = ref_df.groupby('hour')['vol_total'].median()
                if norm_hourly_vol.std() > 0:
                    day_vector = day_df.set_index('hour')['vol_total']
                    if day_vector.std() > 0:
                        corr = np.corrcoef(norm_hourly_vol.values, day_vector.values)[0, 1]
                        if corr < 0.65 and day_vector.sum() > 1000:
                            anomalies_list.append({
                                'Участок дороги': section,
                                'Дата': dt_date,
                                'Уровень': 'Высокий',
                                'Детектор': 'Форма профиля',
                                'Тип аномалии': 'Искажение суточного профиля',
                                'Параметр': 'Корреляция с нормой',
                                'Факт': round(corr, 2),
                                'Норма': '> 0.80',
                                'Отклонение (%)': round((corr - 1.0) * 100, 1),
                                'Индекс (Score)': 2.5,
                                'Причина': _get_cause(dt_date, holidays_dict, day_df),
                                'Описание цифрами': f"Форма суточного хода нетипична (корреляция {corr:.2f} < 0.65)"
                            })

    res_df = pd.DataFrame(anomalies_list)
    if not res_df.empty:
        res_df = res_df.sort_values(by=['Участок дороги', 'Дата'])
    return res_df


def _get_cause(dt_date, holidays_dict, day_df):
    """Определяет причину аномалии."""
    if isinstance(dt_date, pd.Timestamp):
        dt_date = dt_date.date()

    if dt_date in holidays_dict:
        return f"Праздник: {holidays_dict[dt_date]}"

    if day_df['is_weekend'].iloc[0] == 1:
        weekday_name = day_df['timestamp'].iloc[0].strftime('%A')
        return f"Выходной день ({weekday_name})"

    next_day = dt_date + datetime.timedelta(days=1)
    if next_day in holidays_dict:
        return f"Канун праздника: {holidays_dict[next_day]}"

    return "Необъяснимая аномалия — требует ручной проверки"


# ============================================================
# 4. ДЕТЕКТОР АНОМАЛИЙ (МЕСЯЧНЫЕ)
# ============================================================
def detect_monthly_anomalies(df):
    """Поиск аномалий в месячных данных (год к году)."""
    if df.empty:
        return pd.DataFrame()

    df = df.copy()
    df['year'] = df['timestamp'].dt.year
    df['month'] = df['timestamp'].dt.month
    df['year_month'] = df['timestamp'].dt.to_period('M').astype(str)

    monthly_stats = []
    for (sec, ym), m_df in df.groupby(['road_section', 'year_month']):
        y = m_df['year'].iloc[0]
        m = m_df['month'].iloc[0]

        tot_hours = len(m_df)
        days_in_m = pd.Period(ym).days_in_month
        exp_hours = days_in_m * 24
        completeness = (tot_hours / exp_hours) * 100

        daily_vols = m_df.groupby(m_df['timestamp'].dt.date)['vol_total'].sum()
        madt = daily_vols.mean() if len(daily_vols) > 0 else 0

        avg_spd_dir = m_df['speed_direct'].mean()
        tot_vol = m_df['vol_total'].sum()
        tot_heavy = (m_df['large_trucks_total'].sum() +
                     m_df['road_trains_total'].sum())
        heavy_share = (tot_heavy / tot_vol * 100) if tot_vol > 0 else 0

        monthly_stats.append({
            'road_section': sec,
            'year_month': ym,
            'year': y,
            'month': m,
            'completeness_pct': round(completeness, 1),
            'madt': round(madt, 1),
            'avg_speed_dir': round(avg_spd_dir, 1),
            'heavy_share': round(heavy_share, 1)
        })

    m_df = pd.DataFrame(monthly_stats)
    monthly_anomalies = []

    for (sec, cal_month), group in m_df.groupby(['road_section', 'month']):
        if len(group) < 2:
            continue

        madt_med = group['madt'].median()
        spd_med = group['avg_speed_dir'].median()
        hvy_med = group['heavy_share'].median()

        for _, row in group.iterrows():
            ym = row['year_month']

            if row['completeness_pct'] < 75.0:
                monthly_anomalies.append({
                    'Участок дороги': sec,
                    'Период': ym,
                    'Уровень': 'Критический',
                    'Детектор': 'Полнота данных',
                    'Параметр': 'Отработано часов (%)',
                    'Факт': f"{row['completeness_pct']}%",
                    'Норма': '>= 95%',
                    'Отклонение (%)': round(row['completeness_pct'] - 100, 1),
                    'Описание цифрами': f"Данные за месяц неполные (сохранено лишь {row['completeness_pct']}% часов)"
                })

            delta_madt = ((row['madt'] - madt_med) / madt_med) * 100 if madt_med > 0 else 0
            if abs(delta_madt) >= 25.0:
                lvl = 'Критический' if abs(delta_madt) >= 40.0 else 'Высокий'
                monthly_anomalies.append({
                    'Участок дороги': sec,
                    'Период': ym,
                    'Уровень': lvl,
                    'Детектор': 'Межгодовой YoY',
                    'Параметр': 'Среднесуточная интенсивность (ССИ)',
                    'Факт': int(row['madt']),
                    'Норма': int(madt_med),
                    'Отклонение (%)': round(delta_madt, 1),
                    'Описание цифрами': f"ССИ месяца ({int(row['madt'])} авт./сут) отклонилась на {delta_madt:+.1f}% от многолетней нормы ({int(madt_med)})"
                })

            delta_hvy = row['heavy_share'] - hvy_med
            if abs(delta_hvy) >= 8.0:
                monthly_anomalies.append({
                    'Участок дороги': sec,
                    'Период': ym,
                    'Уровень': 'Высокий',
                    'Детектор': 'Состав потока YoY',
                    'Параметр': 'Доля тяжелых грузовиков (%)',
                    'Факт': f"{row['heavy_share']}%",
                    'Норма': f"{hvy_med}%",
                    'Отклонение (%)': round((delta_hvy / hvy_med) * 100, 1) if hvy_med > 0 else 0,
                    'Описание цифрами': f"Доля тяжелых грузовиков изменилась на {delta_hvy:+.1f}% п.п. (факт: {row['heavy_share']}%, норма: {hvy_med}%)"
                })

            delta_spd = row['avg_speed_dir'] - spd_med
            if delta_spd <= -15.0:
                monthly_anomalies.append({
                    'Участок дороги': sec,
                    'Период': ym,
                    'Уровень': 'Критический',
                    'Детектор': 'Скорость YoY',
                    'Параметр': 'Средняя скорость (км/ч)',
                    'Факт': f"{row['avg_speed_dir']}",
                    'Норма': f"{spd_med}",
                    'Отклонение (%)': round((delta_spd / spd_med) * 100, 1) if spd_med > 0 else 0,
                    'Описание цифрами': f"Средняя скорость за месяц упала на {abs(delta_spd):.1f} км/ч ниже многолетней нормы"
                })

    res_df = pd.DataFrame(monthly_anomalies)
    if not res_df.empty:
        res_df = res_df.sort_values(by=['Участок дороги', 'Период'])
    return res_df


# ============================================================
# 5. ГЕНЕРАЦИЯ ГРАФИКОВ
# ============================================================
def generate_anomaly_plots(df, daily_anomalies_df, output_folder):
    """Генерирует графики для каждого аномального дня."""
    os.makedirs(output_folder, exist_ok=True)
    if daily_anomalies_df.empty or df.empty:
        print("   Нет данных для генерации графиков")
        return

    df = df.copy()
    df['date'] = df['timestamp'].dt.date
    df['hour'] = df['timestamp'].dt.hour
    df['is_weekend'] = df['timestamp'].dt.weekday.isin([5, 6]).astype(int)
    df['month_key'] = df['timestamp'].dt.to_period('M').astype(str)

    grouped = daily_anomalies_df.groupby(['Участок дороги', 'Дата'])
    print(f"\n   Генерация графиков (уникальных аномальных дней: {len(grouped)})...")

    hours = np.arange(24)
    plot_count = 0

    for (sec, dt_d), rows in grouped:
        day_df = df[(df['road_section'] == sec) & (df['date'] == dt_d)].sort_values('hour')
        if len(day_df) == 0:
            continue

        m_key = day_df['month_key'].iloc[0]
        is_wknd = day_df['is_weekend'].iloc[0]

        anom_types = "; ".join(rows['Тип аномалии'].unique())
        causes = "; ".join(rows['Причина'].unique())

        ref_df = df[(df['road_section'] == sec) &
                   (df['month_key'] == m_key) &
                   (df['is_weekend'] == is_wknd)]
        norm_vol = ref_df.groupby('hour')['vol_total'].median().reindex(hours).fillna(0).values
        norm_spd = ref_df.groupby('hour')['speed_direct'].median().reindex(hours).fillna(0).values

        fact_vol = day_df.set_index('hour')['vol_total'].reindex(hours).fillna(0).values
        fact_spd = day_df.set_index('hour')['speed_direct'].reindex(hours).fillna(0).values

        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True,
                                        gridspec_kw={'height_ratios': [2, 1]})

        ax1.bar(hours, norm_vol, color='#BDC3C7', alpha=0.6, width=0.6,
                label='Норма месяца (Медиана)', edgecolor='#95A5A6')
        ax1.plot(hours, fact_vol, color='#E74C3C', linewidth=2.5, marker='o',
                 markersize=5, label='Факт аномального дня')
        ax1.set_ylabel('Интенсивность, авт./ч', fontsize=11, fontweight='bold')
        ax1.set_title(f"{sec[:80]}\n{dt_d} | События: {anom_types}\nПричина: {causes}",
                      fontsize=11, fontweight='bold', pad=12)
        ax1.grid(True, linestyle='--', alpha=0.5)
        ax1.legend(loc='upper left', frameon=True)

        diff = np.abs(fact_vol - norm_vol)
        max_h = np.argmax(diff)
        if diff[max_h] > 100:
            y_pos = fact_vol[max_h] * 1.15 if fact_vol[max_h] > norm_vol[max_h] else fact_vol[max_h] * 0.7
            ax1.annotate(f'Пик: {int(fact_vol[max_h])} vs {int(norm_vol[max_h])}',
                         xy=(max_h, fact_vol[max_h]),
                         xytext=(max_h, y_pos),
                         arrowprops=dict(facecolor='black', shrink=0.05, width=1, headwidth=5),
                         fontsize=9, fontweight='bold',
                         bbox=dict(boxstyle="round,pad=0.3", fc="#FADBD8", ec="#E74C3C", lw=1))

        ax2.plot(hours, norm_spd, color='#7F8C8D', linestyle='--', linewidth=1.8,
                 label='Норма скорости')
        ax2.plot(hours, fact_spd, color='#2980B9', linewidth=2.0, marker='s',
                 markersize=4, label='Факт скорости (Прямое)')
        ax2.axhline(40, color='#C0392B', linestyle=':', alpha=0.7, label='Порог затора (40 км/ч)')
        ax2.set_ylabel('Скорость, км/ч', fontsize=11, fontweight='bold')
        ax2.set_xlabel('Час суток (00:00 - 23:00)', fontsize=11, fontweight='bold')
        ax2.set_xticks(hours)
        ax2.set_xticklabels([f"{h:02d}" for h in hours])
        ax2.grid(True, linestyle='--', alpha=0.5)
        ax2.legend(loc='lower left', frameon=True)
        ax2.set_ylim(0, max(110, np.max(fact_spd) + 10))

        plt.tight_layout()
        safe_sec = re.sub(r'[^a-zA-Z0-9а-яА-Я+]', '_', sec)[:30]
        fname = f"{dt_d}_{safe_sec}.png"
        fig.savefig(os.path.join(output_folder, fname), dpi=120, bbox_inches='tight')
        plt.close(fig)
        plot_count += 1

    print(f"   Сохранено {plot_count} графиков")


# ============================================================
# 6. ЭКСПОРТ В EXCEL
# ============================================================
def export_anomalies_to_excel(daily_df, monthly_df, summary_df, output_path):
    """Экспортирует результаты в форматированный Excel-файл."""
    with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
        if not daily_df.empty:
            daily_df.to_excel(writer, sheet_name="Аномальные сутки", index=False)
        else:
            pd.DataFrame([{"Инфо": "Аномальные сутки не обнаружены"}]).to_excel(
                writer, sheet_name="Аномальные сутки", index=False)

        if not monthly_df.empty:
            monthly_df.to_excel(writer, sheet_name="Аномальные месяцы (YoY)", index=False)
        else:
            pd.DataFrame([{"Инфо": "Аномальные месяцы не обнаружены"}]).to_excel(
                writer, sheet_name="Аномальные месяцы (YoY)", index=False)

        if not summary_df.empty:
            summary_df.to_excel(writer, sheet_name="Сводная статистика", index=False)
        else:
            pd.DataFrame([{"Инфо": "Нет данных для сводки"}]).to_excel(
                writer, sheet_name="Сводная статистика", index=False)

    wb = openpyxl.load_workbook(output_path)

    header_fill = PatternFill(start_color="1F497D", end_color="1F497D", fill_type="solid")
    header_font = Font(name="Calibri", size=11, bold=True, color="FFFFFF")
    crit_fill = PatternFill(start_color="FCE4D6", end_color="FCE4D6", fill_type="solid")
    high_fill = PatternFill(start_color="FFF2CC", end_color="FFF2CC", fill_type="solid")
    border_thin = Border(
        left=Side(style='thin', color='D9D9D9'),
        right=Side(style='thin', color='D9D9D9'),
        top=Side(style='thin', color='D9D9D9'),
        bottom=Side(style='thin', color='D9D9D9')
    )

    for ws in wb.worksheets:
        for col_idx in range(1, ws.max_column + 1):
            cell = ws.cell(row=1, column=col_idx)
            cell.fill = header_fill
            cell.font = header_font
            cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)

        for row_idx in range(2, ws.max_row + 1):
            level_val = None
            for col_idx in range(1, ws.max_column + 1):
                c = ws.cell(row=row_idx, column=col_idx)
                c.border = border_thin
                c.alignment = Alignment(vertical="center")
                if ws.cell(row=1, column=col_idx).value == 'Уровень':
                    level_val = c.value

            if level_val == 'Критический':
                for col_idx in range(1, ws.max_column + 1):
                    ws.cell(row=row_idx, column=col_idx).fill = crit_fill
            elif level_val == 'Высокий':
                for col_idx in range(1, ws.max_column + 1):
                    ws.cell(row=row_idx, column=col_idx).fill = high_fill

        for col in ws.columns:
            max_len = max(len(str(cell.value or '')) for cell in col)
            col_letter = openpyxl.utils.get_column_letter(col[0].column)
            ws.column_dimensions[col_letter].width = min(max(max_len + 3, 12), 50)

        ws.row_dimensions[1].height = 30

    wb.save(output_path)


# ============================================================
# 7. СОЗДАНИЕ СВОДНОЙ ТАБЛИЦЫ
# ============================================================
def create_summary(daily_df, monthly_df, full_df):
    """Создаёт сводную таблицу по участкам."""
    summary_rows = []

    for section in full_df['road_section'].unique():
        sec_daily = daily_df[daily_df['Участок дороги'] == section] if not daily_df.empty else pd.DataFrame()
        sec_monthly = monthly_df[monthly_df['Участок дороги'] == section] if not monthly_df.empty else pd.DataFrame()
        sec_data = full_df[full_df['road_section'] == section]

        total_days = sec_data['date'].nunique() if 'date' in sec_data.columns else 0
        total_anomalies = len(sec_daily)
        critical_count = len(sec_daily[sec_daily['Уровень'] == 'Критический']) if not sec_daily.empty else 0
        high_count = len(sec_daily[sec_daily['Уровень'] == 'Высокий']) if not sec_daily.empty else 0

        if not sec_daily.empty and 'Причина' in sec_daily.columns:
            top_causes = sec_daily['Причина'].value_counts().head(3).to_dict()
            causes_str = "; ".join([f"{k}: {v}" for k, v in top_causes.items()])
        else:
            causes_str = "Нет данных"

        if not sec_daily.empty and 'Тип аномалии' in sec_daily.columns:
            top_types = sec_daily['Тип аномалии'].value_counts().head(3).to_dict()
            types_str = "; ".join([f"{k}: {v}" for k, v in top_types.items()])
        else:
            types_str = "Нет данных"

        summary_rows.append({
            'Участок дороги': section,
            'Всего дней в базе': total_days,
            'Аномальных дней': total_anomalies,
            'Критических': critical_count,
            'Высоких': high_count,
            'Топ причин': causes_str,
            'Топ типов аномалий': types_str,
            'Аномальных месяцев': len(sec_monthly)
        })

    return pd.DataFrame(summary_rows)


# ============================================================
# 8. MAIN (для Jupyter Notebook)
# ============================================================
# НАСТРОЙКИ ПУТЕЙ (измени под свои пути)
INPUT_FOLDER = "./input_data"
OUTPUT_FOLDER = "./reports"

print(f"\n{'='*60}")
print(f"  АНАЛИЗАТОР АНОМАЛИЙ ТРАФИКА")
print(f"  Период: 01.09.2023 — 01.09.2026")
print(f"{'='*60}")

# Создаём папки
os.makedirs(INPUT_FOLDER, exist_ok=True)
os.makedirs(OUTPUT_FOLDER, exist_ok=True)
plots_folder = os.path.join(OUTPUT_FOLDER, "plots")
os.makedirs(plots_folder, exist_ok=True)

# Ищем Excel-файлы
excel_files = sorted(
    glob.glob(os.path.join(INPUT_FOLDER, "*.xlsx")) +
    glob.glob(os.path.join(INPUT_FOLDER, "*.xls"))
)

if not excel_files:
    print(f"\n[Внимание] В папке '{INPUT_FOLDER}' нет файлов .xlsx!")
    print("Скопируйте файлы отчетов в эту папку и запусти код заново.")
else:
    print(f"\n[1/6] Загрузка календаря праздников...")
    holidays_dict = get_russia_holidays(range(2023, 2027))
    print(f"   Загружено {len(holidays_dict)} праздничных дней")

    print(f"\n[2/6] Чтение данных из Excel...")
    print(f"   Найдено файлов: {len(excel_files)}")
    all_dfs = []
    for idx, fpath in enumerate(excel_files, 1):
        fname = os.path.basename(fpath)
        print(f"   [{idx}/{len(excel_files)}] {fname}...", end=" ", flush=True)
        try:
            df_part = parse_traffic_excel(fpath)
            print(f"OK ({len(df_part)} строк)")
            all_dfs.append(df_part)
        except Exception as e:
            print(f"ОШИБКА: {e}")
            import traceback
            traceback.print_exc()

    if not all_dfs:
        print("Не удалось прочитать ни одного файла.")
    else:
        full_df = pd.concat(all_dfs, ignore_index=True)
        full_df['date'] = full_df['timestamp'].dt.date
        print(f"\n   Всего загружено: {len(full_df)} записей")
        print(f"   Участки: {full_df['road_section'].nunique()}")
        for sec in full_df['road_section'].unique():
            sec_count = len(full_df[full_df['road_section'] == sec])
            print(f"     - {sec[:70]}... ({sec_count} записей)")

        print("\n[3/6] Поиск аномальных суток...")
        daily_anomalies = detect_daily_anomalies(full_df, holidays_dict)
        print(f"   Найдено аномальных суток: {len(daily_anomalies)}")

        print("\n[4/6] Поиск аномальных месяцев (YoY)...")
        monthly_anomalies = detect_monthly_anomalies(full_df)
        print(f"   Найдено аномальных месяцев: {len(monthly_anomalies)}")

        print("\n[5/6] Создание сводной таблицы...")
        summary_df = create_summary(daily_anomalies, monthly_anomalies, full_df)

        report_xlsx_path = os.path.join(OUTPUT_FOLDER, "traffic_anomalies_report.xlsx")
        export_anomalies_to_excel(daily_anomalies, monthly_anomalies, summary_df, report_xlsx_path)
        print(f"   Реестр сохранен: {report_xlsx_path}")

        print("\n[6/6] Генерация графиков...")
        generate_anomaly_plots(full_df, daily_anomalies, plots_folder)

        print(f"\n{'='*60}")
        print(f"  АНАЛИЗ ПОЛНОСТЬЮ ЗАВЕРШЕН")
        print(f"  Отчёт: {report_xlsx_path}")
        print(f"  Графики: {plots_folder}")
        print(f"{'='*60}")


  АНАЛИЗАТОР АНОМАЛИЙ ТРАФИКА
  Период: 01.09.2023 — 01.09.2026

[1/6] Загрузка календаря праздников...
   Загружено 74 праздничных дней

[2/6] Чтение данных из Excel...
   Найдено файлов: 1
   [1/1] Svodniy_otchet_po_chasam_intensivnost+skorost_01.09.2023_01.09.2026_1792228142.xlsx...    Загружено 41136 записей для 3 участков
OK (41136 строк)

   Всего загружено: 41136 записей
   Участки: 2
     - км 448+650 а/д Р-22  "Каспий" а/д М-4 "Дон", подъезд к г. Саратов... (26303 записей)
     - км 478+742 а/д Р-22  "Каспий" а/д М-4 "Дон", подъезд к г. Саратов... (14833 записей)

[3/6] Поиск аномальных суток...

   Анализ участка: км 448+650 а/д Р-22  "Каспий" а/д М-4 "Дон", подъезд к г. Са...

   Анализ участка: км 478+742 а/д Р-22  "Каспий" а/д М-4 "Дон", подъезд к г. Са...
   Найдено аномальных суток: 26

[4/6] Поиск аномальных месяцев (YoY)...
   Найдено аномальных месяцев: 4

[5/6] Создание сводной таблицы...
   Реестр сохранен: ./reports\traffic_anomalies_report.xlsx

[6/6] Генерация г